# 02 · Transformación Bronze → Silver — YelpConnect

**Tipo:** Pipeline de transformación  
**Origen:** `workspace.yelp_bronze`  
**Destino:** `workspace.yelp_silver`  
**Modalidad:** Batch

---

## Objetivo

Transformar las entidades de la capa Bronze en estructuras Silver
conformadas, tipadas y preparadas para Data Quality, analítica y
Trust & Safety.

## Principios

- Bronze conserva el dato original.
- Silver interpreta, tipa, normaliza y conforma.
- Los registros inválidos de Review no se eliminan:
  se envían a cuarentena.
- Los posibles duplicados lógicos de Review se conservan como
  señal potencial para análisis posterior.
- Una señal de duplicidad no representa fraude confirmado.
- Las reglas Trust & Safety NO se ejecutan en este notebook.
- Data Quality Silver se ejecuta posteriormente como una Task separada.
- Este notebook NO crea infraestructura del Lakehouse.

## Salidas

- `slv_business`
- `slv_user`
- `slv_review`
- `slv_review_cuarentena`
- `slv_tip`
- `slv_checkin_event`


In [0]:
# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

import json
import uuid

from datetime import (
    datetime,
    timezone
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:

# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "yelp_bronze"
SILVER_SCHEMA = "yelp_silver"

BRONZE = (
    f"{CATALOG}."
    f"{BRONZE_SCHEMA}"
)

SILVER = (
    f"{CATALOG}."
    f"{SILVER_SCHEMA}"
)

PIPELINE_NAME = (
    "E7_02_silver_transformacion"
)

SILVER_EXECUTION_ID = str(
    uuid.uuid4()
)

EXECUTION_STARTED_AT = (
    datetime.now(
        timezone.utc
    )
)

REQUIRED_BRONZE_TABLES = [
    f"{BRONZE}.brz_business",
    f"{BRONZE}.brz_user",
    f"{BRONZE}.brz_review",
    f"{BRONZE}.brz_tip",
    f"{BRONZE}.brz_checkin"
]

EXPECTED_SILVER_TABLES = [
    f"{SILVER}.slv_business",
    f"{SILVER}.slv_user",
    f"{SILVER}.slv_review",
    f"{SILVER}.slv_review_cuarentena",
    f"{SILVER}.slv_tip",
    f"{SILVER}.slv_checkin_event"
]

REQUIRED_BRONZE_METADATA = {
    "_archivo_origen",
    "_fecha_ingesta",
    "_source_system",
    "_source_entity",
    "_pipeline_name",
    "_execution_id",
    "_layer",
    "_schema_hash",
    "_record_checksum"
}

REQUIRED_SILVER_METADATA = {
    "_archivo_origen",
    "_bronze_execution_id",
    "_bronze_record_checksum",
    "_silver_execution_id",
    "_silver_processed_at",
    "_layer"
}


print("=" * 72)
print("YELPCONNECT - BRONZE → SILVER")
print("=" * 72)

print(
    f"Pipeline      : {PIPELINE_NAME}"
)

print(
    f"Execution ID  : {SILVER_EXECUTION_ID}"
)

print(
    f"Inicio UTC    : "
    f"{EXECUTION_STARTED_AT.isoformat()}"
)

print(
    f"Origen        : {BRONZE}"
)

print(
    f"Destino       : {SILVER}"
)

print("=" * 72)


# COMMAND ----------


## 1 · Precondiciones

La infraestructura del Lakehouse debe existir antes de ejecutar
este notebook.

Se valida:

- existencia de los schemas Bronze y Silver;
- existencia de las cinco tablas Bronze;
- presencia de metadata técnica Bronze.

Si alguna precondición no se cumple, la ejecución debe detenerse.

In [0]:


# COMMAND ----------

# ============================================================
# 3. VALIDAR SCHEMAS
# ============================================================

for schema_name in [
    BRONZE,
    SILVER
]:

    try:

        spark.sql(
            f"""
            DESCRIBE SCHEMA
            {schema_name}
            """
        )

    except Exception as exc:

        raise RuntimeError(
            f"""
            ❌ SCHEMA REQUERIDO INEXISTENTE

            Schema:
            {schema_name}

            Ejecutar previamente:
            00_setup/01_configurar_lakehouse
            """
        ) from exc


print(
    "✔ Schemas Bronze y Silver disponibles."
)


In [0]:

# COMMAND ----------

# ============================================================
# 4. VALIDAR TABLAS BRONZE
# ============================================================

missing_bronze_tables = [
    table_name
    for table_name
    in REQUIRED_BRONZE_TABLES
    if not spark.catalog.tableExists(
        table_name
    )
]


if missing_bronze_tables:

    raise RuntimeError(
        f"""
        ❌ PRECONDICIÓN SILVER INCUMPLIDA

        Tablas Bronze faltantes:

        {missing_bronze_tables}

        Ejecutar previamente:
        E7_pipeline_lakehouse/01_ingestar_bronze
        """
    )


print(
    "✔ Las 5 tablas Bronze están disponibles."
)


In [0]:

# COMMAND ----------

# ============================================================
# 5. VALIDAR CONTRATO TÉCNICO BRONZE
# ============================================================

for table_name in REQUIRED_BRONZE_TABLES:

    source_df = (
        spark.table(
            table_name
        )
    )

    missing_metadata = (
        REQUIRED_BRONZE_METADATA
        -
        set(
            source_df.columns
        )
    )

    if missing_metadata:

        raise RuntimeError(
            f"""
            ❌ CONTRATO BRONZE INCOMPLETO

            Tabla:
            {table_name}

            Metadata faltante:
            {sorted(missing_metadata)}
            """
        )


print(
    "✔ Metadata técnica Bronze validada."
)


# COMMAND ----------


## 2 · Gobierno de tablas Silver

Las tablas Silver publicadas incorporan metadata descriptiva en
Unity Catalog.

Se aplican:

- descripción funcional de la tabla;
- capa;
- dominio;
- producto de datos;
- sistema origen;
- clasificación;
- criticidad;
- entidad.


In [0]:

# COMMAND ----------

# ============================================================
# 6. FUNCIÓN METADATA UNITY CATALOG
# ============================================================

def apply_silver_uc_metadata(
    spark,
    table_name,
    entity,
    description,
    criticality="medium",
    classification="internal",
    data_product="silver_conformed"
):

    safe_description = (
        description.replace(
            "'",
            "''"
        )
    )

    # --------------------------------------------------------
    # DESCRIPCIÓN
    # --------------------------------------------------------

    spark.sql(
        f"""
        COMMENT ON TABLE {table_name}
        IS '{safe_description}'
        """
    )

    # --------------------------------------------------------
    # TAGS
    # --------------------------------------------------------

    try:

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            SET TAGS (
                'data_layer' = 'silver',
                'data_domain' = 'yelp_source',
                'data_product' = '{data_product}',
                'source_system' = 'yelp',
                'classification' = '{classification}',
                'criticality' = '{criticality}',
                'entity' = '{entity}'
            )
            """
        )

        print(
            f"✔ Tags UC aplicados: "
            f"{table_name}"
        )

    except Exception as exc:

        print(
            f"⚠ No fue posible aplicar tags UC "
            f"a {table_name}"
        )

        print(
            f"Detalle: {exc}"
        )


# COMMAND ----------


# 3 · Business

## Transformaciones

- Normalización de nombre, dirección, ciudad y estado.
- Conversión de coordenadas y métricas a tipos numéricos.
- `is_open` se convierte a boolean.
- `categories` se convierte a `ARRAY<STRING>`.
- `attributes` y `hours` conservan sus estructuras complejas.
- Se excluyen registros sin `business_id`.
- Se deduplica por `business_id`.
- Se preserva linaje hacia Bronze.

**Grano Silver:** una fila por `business_id`.

In [0]:

# COMMAND ----------

# ============================================================
# 7. SILVER BUSINESS
# ============================================================

brz_business = (
    spark.table(
        f"{BRONZE}.brz_business"
    )
)


slv_business = (
    brz_business

    .select(

        F.col(
            "business_id"
        ),

        F.trim(
            F.col("name")
        ).alias(
            "nombre"
        ),

        F.trim(
            F.col("address")
        ).alias(
            "direccion"
        ),

        F.initcap(
            F.trim(
                F.col("city")
            )
        ).alias(
            "ciudad"
        ),

        F.upper(
            F.trim(
                F.col("state")
            )
        ).alias(
            "estado"
        ),

        F.trim(
            F.col(
                "postal_code"
            )
        ).alias(
            "codigo_postal"
        ),

        F.col(
            "latitude"
        ).cast(
            "double"
        ).alias(
            "latitud"
        ),

        F.col(
            "longitude"
        ).cast(
            "double"
        ).alias(
            "longitud"
        ),

        F.col(
            "stars"
        ).cast(
            "double"
        ).alias(
            "estrellas"
        ),

        F.col(
            "review_count"
        ).cast(
            "long"
        ).alias(
            "n_resenas"
        ),

        (
            F.col(
                "is_open"
            ).cast(
                "int"
            )
            ==
            1
        ).alias(
            "esta_abierto"
        ),

        F.when(
            F.col(
                "categories"
            ).isNull(),

            F.lit(
                None
            )
        )
        .otherwise(
            F.split(
                F.col(
                    "categories"
                ),
                r",\s*"
            )
        )
        .alias(
            "categorias"
        ),

        F.col(
            "attributes"
        ).alias(
            "atributos"
        ),

        F.col(
            "hours"
        ).alias(
            "horarios"
        ),

        F.col(
            "_archivo_origen"
        ),

        F.col(
            "_execution_id"
        ).alias(
            "_bronze_execution_id"
        ),

        F.col(
            "_record_checksum"
        ).alias(
            "_bronze_record_checksum"
        ),

        F.lit(
            SILVER_EXECUTION_ID
        ).alias(
            "_silver_execution_id"
        ),

        F.current_timestamp()
        .alias(
            "_silver_processed_at"
        ),

        F.lit(
            "silver"
        ).alias(
            "_layer"
        )
    )

    .filter(
        F.col(
            "business_id"
        ).isNotNull()
    )

    .dropDuplicates(
        [
            "business_id"
        ]
    )
)


(
    slv_business

    .write

    .format(
        "delta"
    )

    .mode(
        "overwrite"
    )

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_business"
    )
)


apply_silver_uc_metadata(
    spark=spark,
    table_name=(
        f"{SILVER}.slv_business"
    ),
    entity="business",
    description=(
        "Entidad Business conformada en Silver, "
        "tipada, normalizada y deduplicada por business_id."
    )
)


business_count = (
    spark.table(
        f"{SILVER}.slv_business"
    )
    .count()
)


print(
    f"✔ slv_business: "
    f"{business_count:,} filas"
)


# COMMAND ----------

# 4 · User

## Transformaciones

- `yelping_since` se convierte a `TIMESTAMP`.
- Contadores se convierten a tipos numéricos.
- `elite` se convierte a `ARRAY<STRING>`.
- `friends` se convierte a `ARRAY<STRING>`.
- Se excluyen registros sin `user_id`.
- Se deduplica por `user_id`.
- Se preserva linaje hacia Bronze.

**Grano Silver:** una fila por `user_id`.

> El dataset User corresponde a una muestra parcial. La ausencia
> de un `user_id` en `slv_user` no implica que una Review sea
> inválida o sospechosa.

In [0]:

# COMMAND ----------

# ============================================================
# 8. FUNCIÓN CSV → ARRAY
# ============================================================

def csv_to_array(
    column_name
):

    return (

        F.when(

            F.col(
                column_name
            ).isNull()

            |

            (
                F.trim(
                    F.col(
                        column_name
                    )
                )
                ==
                ""
            ),

            F.expr(
                "array()"
            ).cast(
                "array<string>"
            )
        )

        .otherwise(

            F.split(
                F.col(
                    column_name
                ),
                r",\s*"
            )
        )
    )

In [0]:

# COMMAND ----------

# ============================================================
# 9. SILVER USER
# ============================================================

brz_user = (
    spark.table(
        f"{BRONZE}.brz_user"
    )
)


slv_user = (
    brz_user

    .select(

        F.col(
            "user_id"
        ),

        F.trim(
            F.col("name")
        ).alias(
            "nombre"
        ),

        F.to_timestamp(
            F.col(
                "yelping_since"
            )
        ).alias(
            "yelping_since"
        ),

        F.col(
            "review_count"
        ).cast(
            "long"
        ).alias(
            "review_count"
        ),

        F.col(
            "useful"
        ).cast(
            "long"
        ).alias(
            "useful"
        ),

        F.col(
            "funny"
        ).cast(
            "long"
        ).alias(
            "funny"
        ),

        F.col(
            "cool"
        ).cast(
            "long"
        ).alias(
            "cool"
        ),

        F.col(
            "fans"
        ).cast(
            "long"
        ).alias(
            "fans"
        ),

        F.col(
            "average_stars"
        ).cast(
            "double"
        ).alias(
            "average_stars"
        ),

        csv_to_array(
            "elite"
        ).alias(
            "elite"
        ),

        csv_to_array(
            "friends"
        ).alias(
            "friends"
        ),

        F.col(
            "_archivo_origen"
        ),

        F.col(
            "_execution_id"
        ).alias(
            "_bronze_execution_id"
        ),

        F.col(
            "_record_checksum"
        ).alias(
            "_bronze_record_checksum"
        ),

        F.lit(
            SILVER_EXECUTION_ID
        ).alias(
            "_silver_execution_id"
        ),

        F.current_timestamp()
        .alias(
            "_silver_processed_at"
        ),

        F.lit(
            "silver"
        ).alias(
            "_layer"
        )
    )

    .filter(
        F.col(
            "user_id"
        ).isNotNull()
    )

    .dropDuplicates(
        [
            "user_id"
        ]
    )
)


(
    slv_user

    .write

    .format(
        "delta"
    )

    .mode(
        "overwrite"
    )

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_user"
    )
)


apply_silver_uc_metadata(
    spark=spark,
    table_name=(
        f"{SILVER}.slv_user"
    ),
    entity="user",
    description=(
        "Entidad User conformada en Silver, "
        "tipada y deduplicada por user_id."
    )
)


user_count = (
    spark.table(
        f"{SILVER}.slv_user"
    )
    .count()
)


print(
    f"✔ slv_user: "
    f"{user_count:,} filas"
)


# COMMAND ----------


# 5 · Review

Review es la entidad principal del caso Trust & Safety.

## Transformaciones

- `stars` se convierte a `DOUBLE`.
- `date` se convierte a `TIMESTAMP`.
- El texto se normaliza mediante `trim`.
- Se verifica existencia de Business.
- Se registra cobertura de User.
- Se detectan duplicados por `review_id`.
- Se identifica posible duplicación lógica.
- Se separan Reviews válidas y Reviews en cuarentena.

## Importante

Un posible duplicado lógico:

- NO se elimina automáticamente;
- NO representa fraude confirmado;
- se conserva como señal heurística para análisis posterior.

La ausencia de `user_id` en la muestra User:

- NO envía automáticamente la Review a cuarentena;
- queda registrada mediante `_user_en_muestra = false`.

In [0]:
# COMMAND ----------

# ============================================================
# 10. REVIEW - FUENTE
# ============================================================

brz_review = (
    spark.table(
        f"{BRONZE}.brz_review"
    )
)


business_keys = (
    spark.table(
        f"{SILVER}.slv_business"
    )

    .select(
        "business_id"
    )

    .dropDuplicates()

    .withColumn(
        "_business_existe",
        F.lit(
            True
        )
    )
)


user_keys = (
    spark.table(
        f"{SILVER}.slv_user"
    )

    .select(
        "user_id"
    )

    .dropDuplicates()

    .withColumn(
        "_user_en_muestra",
        F.lit(
            True
        )
    )
)


In [0]:

# COMMAND ----------

# ============================================================
# 11. CONFORMACIÓN REVIEW
# ============================================================

review_base = (
    brz_review

    .select(

        F.col(
            "review_id"
        ),

        F.col(
            "user_id"
        ),

        F.col(
            "business_id"
        ),

        F.col(
            "stars"
        ).cast(
            "double"
        ).alias(
            "stars"
        ),

        F.trim(
            F.col(
                "text"
            )
        ).alias(
            "review_text"
        ),

        F.to_timestamp(
            F.col(
                "date"
            )
        ).alias(
            "review_ts"
        ),

        F.col(
            "useful"
        ).cast(
            "long"
        ).alias(
            "useful"
        ),

        F.col(
            "funny"
        ).cast(
            "long"
        ).alias(
            "funny"
        ),

        F.col(
            "cool"
        ).cast(
            "long"
        ).alias(
            "cool"
        ),

        F.col(
            "_archivo_origen"
        ),

        F.col(
            "_execution_id"
        ).alias(
            "_bronze_execution_id"
        ),

        F.col(
            "_record_checksum"
        ).alias(
            "_bronze_record_checksum"
        ),

        F.lit(
            SILVER_EXECUTION_ID
        ).alias(
            "_silver_execution_id"
        ),

        F.current_timestamp()
        .alias(
            "_silver_processed_at"
        ),

        F.lit(
            "silver"
        ).alias(
            "_layer"
        )
    )
)




In [0]:
# COMMAND ----------

# ============================================================
# 12. INTEGRIDAD REFERENCIAL REVIEW
# ============================================================

review_integridad = (
    review_base

    .join(
        business_keys,
        "business_id",
        "left"
    )

    .join(
        user_keys,
        "user_id",
        "left"
    )

    .fillna(
        {
            "_business_existe":
                False,

            "_user_en_muestra":
                False
        }
    )
)


In [0]:

# COMMAND ----------

# ============================================================
# 13. DUPLICADO POR REVIEW_ID
# ============================================================

w_review_id = (
    Window

    .partitionBy(
        "review_id"
    )

    .orderBy(
        F.col(
            "_bronze_record_checksum"
        ),
        F.col(
            "_archivo_origen"
        )
    )
)


review_dedup = (
    review_integridad

    .withColumn(
        "_review_id_row_number",

        F.row_number()
        .over(
            w_review_id
        )
    )

    .withColumn(
        "_duplicate_review_id",

        F.col(
            "_review_id_row_number"
        )
        >
        1
    )
)


In [0]:
# COMMAND ----------

# ============================================================
# 14. POSIBLE DUPLICADO LÓGICO
# ============================================================

review_dedup = (
    review_dedup

    .withColumn(
        "_logical_review_hash",

        F.sha2(

            F.concat_ws(

                "||",

                F.coalesce(
                    F.col(
                        "user_id"
                    ),
                    F.lit("")
                ),

                F.coalesce(
                    F.col(
                        "business_id"
                    ),
                    F.lit("")
                ),

                F.lower(
                    F.trim(
                        F.coalesce(
                            F.col(
                                "review_text"
                            ),
                            F.lit("")
                        )
                    )
                )
            ),

            256
        )
    )
)


w_logical = (
    Window
    .partitionBy(
        "_logical_review_hash"
    )
)


review_dedup = (
    review_dedup

    .withColumn(
        "_logical_duplicate_count",

        F.count(
            "*"
        ).over(
            w_logical
        )
    )

    .withColumn(
        "_possible_logical_duplicate",

        F.col(
            "_logical_duplicate_count"
        )
        >
        1
    )
)


In [0]:
# COMMAND ----------

# ============================================================
# 15. CLASIFICACIÓN PARA CUARENTENA
# ============================================================

review_clasificada = (
    review_dedup

    .withColumn(

        "_motivo_cuarentena",

        F.when(
            F.col(
                "review_id"
            ).isNull(),

            F.lit(
                "REVIEW_ID_NULL"
            )
        )

        .when(
            F.col(
                "business_id"
            ).isNull(),

            F.lit(
                "BUSINESS_ID_NULL"
            )
        )

        .when(
            ~F.col(
                "_business_existe"
            ),

            F.lit(
                "BUSINESS_NOT_FOUND"
            )
        )

        .when(
            F.col(
                "stars"
            ).isNull(),

            F.lit(
                "STARS_NULL_OR_INVALID"
            )
        )

        .when(
            ~F.col(
                "stars"
            ).between(
                1.0,
                5.0
            ),

            F.lit(
                "STARS_OUT_OF_RANGE"
            )
        )

        .when(
            F.col(
                "review_ts"
            ).isNull(),

            F.lit(
                "INVALID_REVIEW_DATE"
            )
        )

        .when(
            F.col(
                "_duplicate_review_id"
            ),

            F.lit(
                "DUPLICATE_REVIEW_ID"
            )
        )

        .otherwise(
            F.lit(
                None
            )
        )
    )
)



## 5.1 · Review válida

Una Review es publicada en `slv_review` cuando no presenta
ninguna condición de cuarentena.

Los posibles duplicados lógicos se conservan mediante:

- `_logical_review_hash`
- `_logical_duplicate_count`
- `_possible_logical_duplicate`


In [0]:

# COMMAND ----------

# ============================================================
# 16. REVIEW VÁLIDA
# ============================================================

slv_review = (
    review_clasificada

    .filter(
        F.col(
            "_motivo_cuarentena"
        ).isNull()
    )

    .drop(
        "_review_id_row_number",
        "_motivo_cuarentena",
        "_duplicate_review_id"
    )
)


(
    slv_review

    .write

    .format(
        "delta"
    )

    .mode(
        "overwrite"
    )

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_review"
    )
)


apply_silver_uc_metadata(
    spark=spark,
    table_name=(
        f"{SILVER}.slv_review"
    ),
    entity="review",
    description=(
        "Reviews conformadas y válidas para "
        "Data Quality, analítica y Trust & Safety. "
        "Los posibles duplicados lógicos se conservan "
        "como señales heurísticas."
    ),
    criticality="high"
)




## 5.2 · Review cuarentena

Las Reviews que no cumplen las condiciones mínimas de conformación
se conservan en `slv_review_cuarentena`.

La cuarentena evita perder evidencia del dato original y permite
investigar posteriormente los defectos de calidad.

In [0]:

# COMMAND ----------

# ============================================================
# 17. REVIEW CUARENTENA
# ============================================================

slv_review_cuarentena = (
    review_clasificada

    .filter(
        F.col(
            "_motivo_cuarentena"
        ).isNotNull()
    )
)


(
    slv_review_cuarentena

    .write

    .format(
        "delta"
    )

    .mode(
        "overwrite"
    )

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_review_cuarentena"
    )
)


apply_silver_uc_metadata(
    spark=spark,
    table_name=(
        f"{SILVER}.slv_review_cuarentena"
    ),
    entity="review",
    description=(
        "Reviews separadas del conjunto Silver válido "
        "por defectos de conformación o integridad."
    ),
    criticality="high",
    data_product="review_quarantine"
)


review_count = (
    spark.table(
        f"{SILVER}.slv_review"
    )
    .count()
)


quarantine_count = (
    spark.table(
        f"{SILVER}.slv_review_cuarentena"
    )
    .count()
)


print(
    f"✔ slv_review: "
    f"{review_count:,} filas"
)

print(
    f"⚠ slv_review_cuarentena: "
    f"{quarantine_count:,} filas"
)


# COMMAND ----------


## Importante sobre User

Si una Review contiene un `user_id` que no aparece en
`slv_user`, la Review **no se envía automáticamente a cuarentena**.

El dataset User es una muestra parcial y puede no contener la
totalidad de usuarios presentes en Review.

Esta situación queda registrada mediante:

`_user_en_muestra = false`

y será medida posteriormente por Data Quality Silver.

La ausencia en la muestra User no representa fraude ni una
señal Trust & Safety por sí sola.

# COMMAND ----------


# 6 · Tip

Yelp Tip no posee un `tip_id` nativo.

Se genera un identificador determinístico SHA-256 utilizando:

`user_id + business_id + date + text`

**Grano Silver:** una fila por Tip lógico.


In [0]:

# COMMAND ----------

# ============================================================
# 18. SILVER TIP
# ============================================================

brz_tip = (
    spark.table(
        f"{BRONZE}.brz_tip"
    )
)


slv_tip = (
    brz_tip

    .withColumn(
        "tip_id",

        F.sha2(

            F.concat_ws(

                "||",

                F.coalesce(
                    F.col(
                        "user_id"
                    ),
                    F.lit("")
                ),

                F.coalesce(
                    F.col(
                        "business_id"
                    ),
                    F.lit("")
                ),

                F.coalesce(
                    F.col(
                        "date"
                    ),
                    F.lit("")
                ),

                F.coalesce(
                    F.col(
                        "text"
                    ),
                    F.lit("")
                )
            ),

            256
        )
    )

    .select(

        F.col(
            "tip_id"
        ),

        F.col(
            "user_id"
        ),

        F.col(
            "business_id"
        ),

        F.trim(
            F.col(
                "text"
            )
        ).alias(
            "tip_text"
        ),

        F.to_timestamp(
            F.col(
                "date"
            )
        ).alias(
            "tip_ts"
        ),

        F.col(
            "compliment_count"
        ).cast(
            "long"
        ).alias(
            "compliment_count"
        ),

        F.col(
            "_archivo_origen"
        ),

        F.col(
            "_execution_id"
        ).alias(
            "_bronze_execution_id"
        ),

        F.col(
            "_record_checksum"
        ).alias(
            "_bronze_record_checksum"
        ),

        F.lit(
            SILVER_EXECUTION_ID
        ).alias(
            "_silver_execution_id"
        ),

        F.current_timestamp()
        .alias(
            "_silver_processed_at"
        ),

        F.lit(
            "silver"
        ).alias(
            "_layer"
        )
    )

    .dropDuplicates(
        [
            "tip_id"
        ]
    )
)


(
    slv_tip

    .write

    .format(
        "delta"
    )

    .mode(
        "overwrite"
    )

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_tip"
    )
)


apply_silver_uc_metadata(
    spark=spark,
    table_name=(
        f"{SILVER}.slv_tip"
    ),
    entity="tip",
    description=(
        "Entidad Tip conformada en Silver con "
        "identificador determinístico SHA-256."
    )
)


tip_count = (
    spark.table(
        f"{SILVER}.slv_tip"
    )
    .count()
)


print(
    f"✔ slv_tip: "
    f"{tip_count:,} filas"
)


# COMMAND ----------

# 7 · Check-in

En Bronze:

`1 fila = Business + lista de fechas`

En Silver:

`1 fila = 1 evento Check-in`

La transformación utiliza:

`split → explode → timestamp`

Los timestamps que no puedan interpretarse no se eliminan
silenciosamente. Permanecen como `NULL` para que Data Quality
Silver pueda detectarlos y medirlos.

**Grano Silver:** un evento Check-in.


In [0]:

# COMMAND ----------

# ============================================================
# 19. SILVER CHECK-IN EVENT
# ============================================================

brz_checkin = (
    spark.table(
        f"{BRONZE}.brz_checkin"
    )
)


slv_checkin_event = (
    brz_checkin

    .withColumn(
        "_checkin_raw",

        F.explode(
            F.split(
                F.col(
                    "date"
                ),
                r",\s*"
            )
        )
    )

    .select(

        F.col(
            "business_id"
        ),

        F.to_timestamp(
            F.trim(
                F.col(
                    "_checkin_raw"
                )
            )
        ).alias(
            "checkin_ts"
        ),

        F.col(
            "_archivo_origen"
        ),

        F.col(
            "_execution_id"
        ).alias(
            "_bronze_execution_id"
        ),

        F.col(
            "_record_checksum"
        ).alias(
            "_bronze_record_checksum"
        ),

        F.lit(
            SILVER_EXECUTION_ID
        ).alias(
            "_silver_execution_id"
        ),

        F.current_timestamp()
        .alias(
            "_silver_processed_at"
        ),

        F.lit(
            "silver"
        ).alias(
            "_layer"
        )
    )
)


(
    slv_checkin_event

    .write

    .format(
        "delta"
    )

    .mode(
        "overwrite"
    )

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{SILVER}.slv_checkin_event"
    )
)


apply_silver_uc_metadata(
    spark=spark,
    table_name=(
        f"{SILVER}.slv_checkin_event"
    ),
    entity="checkin",
    description=(
        "Eventos Check-in conformados en Silver. "
        "Cada fila representa un evento temporal "
        "asociado a un business_id."
    )
)


checkin_count = (
    spark.table(
        f"{SILVER}.slv_checkin_event"
    )
    .count()
)


print(
    f"✔ slv_checkin_event: "
    f"{checkin_count:,} eventos"
)


# COMMAND ----------



# 8 · Validación técnica de salidas

Antes de finalizar se comprueba que:

- las seis tablas Silver fueron creadas;
- todas contienen metadata de linaje Silver;
- la ejecución puede ser entregada al siguiente paso.

Esta validación es un control técnico del pipeline.
No reemplaza las reglas formales de Data Quality Silver.

In [0]:

# COMMAND ----------

# ============================================================
# 20. VALIDAR EXISTENCIA DE SALIDAS
# ============================================================

missing_silver_tables = [
    table_name
    for table_name
    in EXPECTED_SILVER_TABLES
    if not spark.catalog.tableExists(
        table_name
    )
]


if missing_silver_tables:

    raise RuntimeError(
        f"""
        ❌ TRANSFORMACIÓN SILVER INCOMPLETA

        Tablas faltantes:

        {missing_silver_tables}
        """
    )


print(
    "✔ Las 6 tablas Silver fueron creadas."
)


In [0]:
# COMMAND ----------

# ============================================================
# 21. VALIDAR METADATA SILVER
# ============================================================

for table_name in EXPECTED_SILVER_TABLES:

    output_df = (
        spark.table(
            table_name
        )
    )

    missing_metadata = (
        REQUIRED_SILVER_METADATA
        -
        set(
            output_df.columns
        )
    )

    if missing_metadata:

        raise RuntimeError(
            f"""
            ❌ METADATA SILVER INCOMPLETA

            Tabla:
            {table_name}

            Columnas faltantes:
            {sorted(missing_metadata)}
            """
        )


print(
    "✔ Metadata Silver validada "
    "para las 6 tablas."
)


# COMMAND ----------


# 9 · Resumen Silver


In [0]:

# COMMAND ----------

# ============================================================
# 22. RESUMEN DE TABLAS
# ============================================================

resultados = [

    (
        "slv_business",
        business_count
    ),

    (
        "slv_user",
        user_count
    ),

    (
        "slv_review",
        review_count
    ),

    (
        "slv_review_cuarentena",
        quarantine_count
    ),

    (
        "slv_tip",
        tip_count
    ),

    (
        "slv_checkin_event",
        checkin_count
    )
]


df_resumen = (
    spark.createDataFrame(
        resultados,
        [
            "tabla",
            "filas"
        ]
    )
)


display(
    df_resumen
)


# COMMAND ----------


## 9.1 · Motivos de cuarentena Review


In [0]:

# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            _motivo_cuarentena,

            COUNT(*) AS total

        FROM
            {SILVER}.slv_review_cuarentena

        GROUP BY
            _motivo_cuarentena

        ORDER BY
            total DESC
        """
    )
)


# COMMAND ----------

## 9.2 · Cobertura User en Review

Esta métrica representa cobertura del dataset User sobre Review.
No representa una regla de fraude.

In [0]:

# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            _user_en_muestra,

            COUNT(*) AS total,

            ROUND(
                COUNT(*) * 100.0
                /
                SUM(
                    COUNT(*)
                ) OVER(),
                2
            ) AS porcentaje

        FROM
            {SILVER}.slv_review

        GROUP BY
            _user_en_muestra

        ORDER BY
            _user_en_muestra DESC
        """
    )
)


# COMMAND ----------


## 9.3 · Posibles duplicados lógicos

Esta métrica representa una señal heurística de similitud.
No representa fraude confirmado.


In [0]:
# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            _possible_logical_duplicate,

            COUNT(*) AS total

        FROM
            {SILVER}.slv_review

        GROUP BY
            _possible_logical_duplicate

        ORDER BY
            _possible_logical_duplicate DESC
        """
    )
)


# COMMAND ----------


## 9.4 · Grano Check-in

In [0]:

# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            COUNT(*) AS total_eventos,

            COUNT(
                DISTINCT business_id
            ) AS negocios,

            MIN(
                checkin_ts
            ) AS primer_checkin,

            MAX(
                checkin_ts
            ) AS ultimo_checkin

        FROM
            {SILVER}.slv_checkin_event
        """
    )
)


# COMMAND ----------


# Definition of Done — Silver

- [x] Infraestructura Lakehouse validada.
- [x] Cinco fuentes Bronze validadas.
- [x] Metadata Bronze validada.
- [x] `slv_business` creada.
- [x] `slv_user` creada.
- [x] `slv_review` creada.
- [x] `slv_review_cuarentena` creada.
- [x] `slv_tip` creada.
- [x] `slv_checkin_event` creada.
- [x] Tipos de datos conformados.
- [x] `review_id` deduplicado.
- [x] Duplicados lógicos detectados sin considerarlos fraude confirmado.
- [x] Integridad Business evaluada para Review.
- [x] Cobertura User registrada.
- [x] Check-ins llevados a grano evento.
- [x] Linaje Bronze preservado.
- [x] Metadata Unity Catalog aplicada.
- [x] Salidas Silver validadas.

---

## Siguiente paso

Ejecutar:

`E5_calidad_datos/03_ejecutar_dq_silver`

In [0]:

# COMMAND ----------

# ============================================================
# 23. RESULTADO FINAL
# ============================================================

EXECUTION_FINISHED_AT = (
    datetime.now(
        timezone.utc
    )
)


summary = {

    "status":
        "SUCCESS",

    "pipeline":
        PIPELINE_NAME,

    "execution_id":
        SILVER_EXECUTION_ID,

    "started_at":
        EXECUTION_STARTED_AT
        .isoformat(),

    "finished_at":
        EXECUTION_FINISHED_AT
        .isoformat(),

    "tables_created":
        len(
            EXPECTED_SILVER_TABLES
        ),

    "business":
        business_count,

    "user":
        user_count,

    "review":
        review_count,

    "review_quarantine":
        quarantine_count,

    "tip":
        tip_count,

    "checkin_event":
        checkin_count
}


print()
print("=" * 72)
print("BRONZE → SILVER FINALIZADO")
print("=" * 72)

print(
    json.dumps(
        summary,
        indent=2
    )
)

print("=" * 72)


dbutils.notebook.exit(
    json.dumps(
        summary
    )
)